# Baseline: Aggregates + CatBoost

Kaggle notebook for low-label comparison against `kaggle_main.ipynb`. It uses the same entity split protocol and reports per-seed, mean, and std rows per label fraction.

In [ ]:
# Cell 1 - Setup
import json
import pathlib
import re
import subprocess
import sys
import warnings

warnings.filterwarnings("ignore")

subprocess.run(
    [sys.executable, "-m", "pip", "install", "-e", "/kaggle/working/denoising-event-sequences", "--quiet"],
    check=True,
)
sys.path.insert(0, "/kaggle/working/denoising-event-sequences")

try:
    from catboost import CatBoostClassifier, Pool
except ImportError:
    subprocess.run([sys.executable, "-m", "pip", "install", "catboost", "--quiet"], check=True)
    from catboost import CatBoostClassifier, Pool

import numpy as np
import pandas as pd
from sklearn.metrics import (
    accuracy_score,
    average_precision_score,
    balanced_accuracy_score,
    f1_score,
    roc_auc_score,
)

from src.data.splits import make_entity_splits

SMOKE_RUN = False
print("Setup complete")


In [ ]:
# Cell 2 - Paths and config
WORKING_DIR = pathlib.Path("/kaggle/working")
DATA_DIR = WORKING_DIR / "data"
OUTPUT_DIR = WORKING_DIR / "outputs"
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)

USE_AMP = False
config = {
    "data": {
        "max_seq_len": 256,
        "min_seq_len": 2,
        "event_type_col": "MCC",
        "timestamp_col": "TRDATETIME",
        "target_col": "target_flag",
        "numerical_cols": ["amount"],
        "categorical_cols": ["channel_type", "trx_category"],
        "group_col": "cl_id",
        "truncation_pretrain": "sliding_window",
        "truncation_eval": "last_events",
        "amount_transform": "robust_scaler",
        "time_transform": "none",
        "train_ratio": 0.70,
        "val_ratio": 0.15,
        "test_ratio": 0.15,
        "min_vocab_count": 5,
    },
    "model": {
        "event_type_emb_dim": 64,
        "cat_emb_dim": 32,
        "num_projection_dim": 64,
        "time_projection_dim": 64,
        "hidden_dim": 256,
        "num_layers": 4,
        "num_heads": 8,
        "dim_feedforward": 1024,
        "dropout": 0.10,
        "activation": "gelu",
        "max_seq_len": 256,
    },
    "pooling": {"type": "gated_attention"},
    "training": {
        "batch_size": 128,
        "num_epochs_finetune": 20,
        "lr": 3e-4,
        "lr_encoder": 3e-5,
        "weight_decay": 0.01,
        "warmup_ratio": 0.05,
        "gradient_clip_val": 1.0,
        "mixed_precision": USE_AMP,
        "early_stopping_patience": 5,
        "log_every_n_steps": 50,
    },
}

LABEL_FRACTIONS = [0.05, 0.25, 1.00]
LABEL_SAMPLING_SEEDS = [42, 43, 44]
METRIC_COLS = ["accuracy", "macro_f1", "weighted_f1", "roc_auc", "pr_auc", "balanced_accuracy"]

print("Output dir:", OUTPUT_DIR)

In [ ]:
# Cell 3 - Load data and shared splits
GROUP_COL = config["data"]["group_col"]
TARGET_COL = config["data"]["target_col"]
TIME_COL = config["data"]["timestamp_col"]
EVENT_COL = config["data"]["event_type_col"]
NUM_COL = "amount"
CAT_COLS = config["data"]["categorical_cols"]

df = pd.read_csv(DATA_DIR / "rosbank" / "train.csv.gz")
df[TIME_COL] = pd.to_datetime(df[TIME_COL], format="%d%b%y:%H:%M:%S")
labels_by_entity = get_entity_labels(df, GROUP_COL, TARGET_COL) if 'get_entity_labels' in globals() else df.groupby(GROUP_COL)[TARGET_COL].first().to_dict()

splits = make_entity_splits(
    df,
    entity_col=GROUP_COL,
    target_col=TARGET_COL,
    train_ratio=config["data"]["train_ratio"],
    val_ratio=config["data"]["val_ratio"],
    test_ratio=config["data"]["test_ratio"],
    seed=42,
)
all_entity_ids = splits["train"] + splits["val"] + splits["test"]
print(df.shape)
print({k: len(v) for k, v in splits.items()})


In [ ]:
# Cell 4 - Shared sampling, metrics, and aggregation helpers
def get_entity_labels(df, group_col, target_col):
    return df.groupby(group_col)[target_col].first().to_dict()


def sample_labeled_entities(train_ids, labels_by_entity, fraction, seed):
    train_ids = list(train_ids)
    if fraction >= 1.0:
        return train_ids

    rng = np.random.default_rng(seed)
    by_label = {}
    for entity_id in train_ids:
        label = int(labels_by_entity[entity_id])
        by_label.setdefault(label, []).append(entity_id)

    sampled = []
    for label, ids in sorted(by_label.items()):
        ids = np.array(ids, dtype=object)
        n = max(1, int(round(len(ids) * fraction)))
        n = min(n, len(ids))
        sampled.extend(rng.choice(ids, size=n, replace=False).tolist())

    rng.shuffle(sampled)
    return sampled


def compute_binary_metrics(y_true, pos_proba):
    y_true = np.asarray(y_true).astype(int)
    pos_proba = np.asarray(pos_proba).astype(float)
    y_pred = (pos_proba >= 0.5).astype(int)
    return {
        "accuracy": float(accuracy_score(y_true, y_pred)),
        "macro_f1": float(f1_score(y_true, y_pred, average="macro", zero_division=0)),
        "weighted_f1": float(f1_score(y_true, y_pred, average="weighted", zero_division=0)),
        "roc_auc": float(roc_auc_score(y_true, pos_proba)),
        "pr_auc": float(average_precision_score(y_true, pos_proba)),
        "balanced_accuracy": float(balanced_accuracy_score(y_true, y_pred)),
    }


def append_mean_std_rows(rows, fractions, metric_cols=METRIC_COLS):
    final_rows = []
    for fraction in fractions:
        runs = [r for r in rows if float(r["fraction"]) == float(fraction)]
        final_rows.extend(runs)
        mean_row = {"fraction": fraction, "seed": "mean", "n_seeds": len(runs)}
        std_row = {"fraction": fraction, "seed": "std", "n_seeds": len(runs)}
        for metric in metric_cols:
            vals = [float(r[metric]) for r in runs if metric in r and pd.notna(r[metric])]
            if vals:
                mean_row[metric] = float(np.mean(vals))
                std_row[metric] = float(np.std(vals, ddof=0))
        final_rows.append(mean_row)
        final_rows.append(std_row)
    return final_rows


def save_results(rows, stem):
    results_df = pd.DataFrame(rows)
    csv_path = OUTPUT_DIR / f"{stem}.csv"
    json_path = OUTPUT_DIR / f"{stem}.json"
    results_df.to_csv(csv_path, index=False)
    json_path.write_text(json.dumps(rows, indent=2, ensure_ascii=False))
    print(f"Saved: {csv_path}")
    print(f"Saved: {json_path}")
    return results_df


In [ ]:
# Cell 5 - Aggregate feature builder
def safe_name(value):
    return re.sub(r"[^0-9A-Za-z_]+", "_", str(value)).strip("_")[:60]


def safe_divide(num, denom):
    return num.div(denom.replace(0, np.nan), axis=0).replace([np.inf, -np.inf], np.nan).fillna(0.0)


def entropy_from_counts(counts):
    totals = counts.sum(axis=1).replace(0, np.nan)
    probs = counts.div(totals, axis=0).fillna(0.0)
    log_probs = np.log(probs.replace(0.0, np.nan)).fillna(0.0)
    return -(probs * log_probs).sum(axis=1)


def add_entropy_features(features, df_source, col, prefix):
    if col not in df_source.columns:
        return features
    counts = pd.crosstab(df_source[GROUP_COL], df_source[col].astype(str)).reindex(features.index, fill_value=0)
    totals = counts.sum(axis=1).replace(0, np.nan)
    sorted_counts = np.sort(counts.to_numpy(), axis=1)[:, ::-1]
    features[f"{prefix}_entropy"] = entropy_from_counts(counts)
    features[f"{prefix}_top1_share"] = pd.Series(sorted_counts[:, :1].sum(axis=1), index=features.index).div(totals).fillna(0.0)
    features[f"{prefix}_top3_share"] = pd.Series(sorted_counts[:, :3].sum(axis=1), index=features.index).div(totals).fillna(0.0)
    return features


def fit_top_values(df_train, col, top_n):
    if col not in df_train.columns:
        return []
    return df_train[col].astype(str).value_counts().head(top_n).index.tolist()


def add_top_count_features(features, df_source, col, top_values, prefix=None, add_counts=True, denom_col="event_count"):
    if not top_values:
        return features
    prefix = prefix or col
    tmp = df_source[[GROUP_COL, col]].copy()
    tmp[col] = tmp[col].astype(str)
    tmp = tmp[tmp[col].isin(top_values)]
    counts = pd.crosstab(tmp[GROUP_COL], tmp[col])
    counts = counts.reindex(index=features.index, columns=top_values, fill_value=0)
    counts.columns = [f"{prefix}_top_{safe_name(v)}_cnt" for v in top_values]
    denom = features[denom_col] if denom_col in features.columns else features["event_count"]
    freqs = counts.div(denom.replace(0, np.nan), axis=0).fillna(0.0)
    freqs.columns = [c.replace("_cnt", "_freq") for c in counts.columns]
    return features.join(counts if add_counts else pd.DataFrame(index=features.index)).join(freqs)


def add_subset_aggregates(features, df_source, prefix):
    if df_source.empty:
        return features
    g = df_source.groupby(GROUP_COL, sort=False)
    sub = g.agg(
        **{
            f"{prefix}_event_count": (EVENT_COL, "size"),
            f"{prefix}_amount_sum": (NUM_COL, "sum"),
            f"{prefix}_amount_mean": (NUM_COL, "mean"),
            f"{prefix}_amount_std": (NUM_COL, "std"),
            f"{prefix}_amount_abs_sum": ("amount_abs", "sum"),
            f"{prefix}_amount_abs_mean": ("amount_abs", "mean"),
            f"{prefix}_positive_share_num": ("is_pos", "mean"),
            f"{prefix}_negative_share_num": ("is_neg", "mean"),
            f"{prefix}_unique_event_types": (EVENT_COL, "nunique"),
            f"{prefix}_active_days": ("date", "nunique"),
        }
    )
    return features.join(sub.reindex(features.index))


def build_aggregate_features(df, all_ids, train_ids):
    working = df.copy()
    working = working.sort_values([GROUP_COL, TIME_COL], kind="stable")
    working[NUM_COL] = pd.to_numeric(working[NUM_COL], errors="coerce").fillna(0.0)
    working["amount_abs"] = working[NUM_COL].abs()
    working["amount_pos"] = working[NUM_COL].clip(lower=0.0)
    working["amount_neg"] = working[NUM_COL].clip(upper=0.0)
    working["is_pos"] = (working[NUM_COL] > 0).astype(int)
    working["is_neg"] = (working[NUM_COL] < 0).astype(int)
    working["date"] = working[TIME_COL].dt.date
    working["gap_days"] = working.groupby(GROUP_COL, sort=False)[TIME_COL].diff().dt.total_seconds().div(86400).fillna(0.0).clip(lower=0.0)
    working["entity_last_time"] = working.groupby(GROUP_COL, sort=False)[TIME_COL].transform("max")
    working["days_before_last"] = working["entity_last_time"].sub(working[TIME_COL]).dt.total_seconds().div(86400).fillna(0.0).clip(lower=0.0)
    working["event_pos"] = working.groupby(GROUP_COL, sort=False).cumcount()
    working["entity_event_count"] = working.groupby(GROUP_COL, sort=False)[EVENT_COL].transform("size")
    working["is_last_half"] = working["event_pos"] >= (working["entity_event_count"] / 2.0)
    working["prev_event"] = working.groupby(GROUP_COL, sort=False)[EVENT_COL].shift(1).astype(str)
    working["transition"] = working["prev_event"] + "__" + working[EVENT_COL].astype(str)
    working.loc[working["prev_event"].eq("nan"), "transition"] = np.nan

    train_set = set(train_ids)
    train_df = working[working[GROUP_COL].isin(train_set)]
    reference_time = train_df[TIME_COL].max()

    g = working.groupby(GROUP_COL, sort=False)
    features = g.agg(
        event_count=(EVENT_COL, "size"),
        amount_sum=(NUM_COL, "sum"),
        amount_mean=(NUM_COL, "mean"),
        amount_std=(NUM_COL, "std"),
        amount_min=(NUM_COL, "min"),
        amount_max=(NUM_COL, "max"),
        amount_median=(NUM_COL, "median"),
        amount_abs_sum=("amount_abs", "sum"),
        amount_abs_mean=("amount_abs", "mean"),
        amount_abs_max=("amount_abs", "max"),
        amount_pos_sum=("amount_pos", "sum"),
        amount_neg_sum=("amount_neg", "sum"),
        positive_event_count=("is_pos", "sum"),
        negative_event_count=("is_neg", "sum"),
        first_time=(TIME_COL, "min"),
        last_time=(TIME_COL, "max"),
        active_days=("date", "nunique"),
        unique_event_types=(EVENT_COL, "nunique"),
        gap_days_mean=("gap_days", "mean"),
        gap_days_std=("gap_days", "std"),
        gap_days_median=("gap_days", "median"),
        gap_days_max=("gap_days", "max"),
        recent_gap_days=("gap_days", "last"),
    )

    for col in CAT_COLS:
        if col in working.columns:
            features[f"unique_{col}"] = g[col].nunique()

    features["time_span_days"] = (
        features["last_time"] - features["first_time"]
    ).dt.total_seconds().div(86400).fillna(0.0)
    features["events_per_active_day"] = features["event_count"] / features["active_days"].replace(0, np.nan)
    features["events_per_span_day"] = features["event_count"] / (features["time_span_days"] + 1.0)
    features["positive_event_share"] = features["positive_event_count"] / features["event_count"].replace(0, np.nan)
    features["negative_event_share"] = features["negative_event_count"] / features["event_count"].replace(0, np.nan)
    features["days_since_last_event_global"] = (reference_time - features["last_time"]).dt.total_seconds().div(86400).clip(lower=0.0).fillna(0.0)
    features = features.drop(columns=["first_time", "last_time"])

    top_specs = [(EVENT_COL, 60), ("channel_type", 20), ("trx_category", 30)]
    fitted_top_values = []
    for col, top_n in top_specs:
        top_values = fit_top_values(train_df, col, top_n)
        fitted_top_values.append((col, top_values))
        features = add_top_count_features(features, working, col, top_values)

    features = add_entropy_features(features, working, EVENT_COL, "event_type")
    for col in CAT_COLS:
        features = add_entropy_features(features, working, col, col)

    for days in [7, 14, 30, 60, 90]:
        prefix = f"last_{days}d"
        subset = working[working["days_before_last"] <= days]
        features = add_subset_aggregates(features, subset, prefix)
        denom_col = f"{prefix}_event_count"
        for col, top_values in fitted_top_values:
            features = add_top_count_features(
                features, subset, col, top_values, prefix=f"{prefix}_{col}", add_counts=False, denom_col=denom_col
            )

    for n_tail in [5, 10, 25, 50]:
        prefix = f"tail_{n_tail}tx"
        subset = working.groupby(GROUP_COL, sort=False).tail(n_tail)
        features = add_subset_aggregates(features, subset, prefix)
        denom_col = f"{prefix}_event_count"
        for col, top_values in fitted_top_values:
            features = add_top_count_features(
                features, subset, col, top_values, prefix=f"{prefix}_{col}", add_counts=False, denom_col=denom_col
            )

    first_half = working[~working["is_last_half"]]
    last_half = working[working["is_last_half"]]
    drift_base = features.copy()
    drift_base = add_subset_aggregates(drift_base, first_half, "first_half")
    drift_base = add_subset_aggregates(drift_base, last_half, "last_half")
    for metric in ["event_count", "amount_sum", "amount_mean", "amount_abs_sum", "positive_share_num", "negative_share_num", "unique_event_types", "active_days"]:
        a = drift_base.get(f"last_half_{metric}", pd.Series(0.0, index=features.index)).fillna(0.0)
        b = drift_base.get(f"first_half_{metric}", pd.Series(0.0, index=features.index)).fillna(0.0)
        features[f"drift_{metric}"] = a - b
    for col, top_values in fitted_top_values:
        first_base = pd.DataFrame({"event_count": first_half.groupby(GROUP_COL).size().reindex(features.index).fillna(0.0)}, index=features.index)
        last_base = pd.DataFrame({"event_count": last_half.groupby(GROUP_COL).size().reindex(features.index).fillna(0.0)}, index=features.index)
        first_tmp = add_top_count_features(first_base, first_half, col, top_values, prefix=f"first_half_{col}", add_counts=False)
        last_tmp = add_top_count_features(last_base, last_half, col, top_values, prefix=f"last_half_{col}", add_counts=False)
        for value in top_values:
            first_col = f"first_half_{col}_top_{safe_name(value)}_freq"
            last_col = f"last_half_{col}_top_{safe_name(value)}_freq"
            features[f"drift_{col}_top_{safe_name(value)}_freq"] = last_tmp[last_col].fillna(0.0) - first_tmp[first_col].fillna(0.0)

    top_transitions = train_df["transition"].dropna().value_counts().head(50).index.tolist()
    if top_transitions:
        trans = working.dropna(subset=["transition"])
        trans = trans[trans["transition"].isin(top_transitions)]
        trans_counts = pd.crosstab(trans[GROUP_COL], trans["transition"]).reindex(index=features.index, columns=top_transitions, fill_value=0)
        trans_counts.columns = [f"transition_top_{safe_name(v)}_cnt" for v in top_transitions]
        trans_freqs = trans_counts.div(features["event_count"].replace(0, np.nan), axis=0).fillna(0.0)
        trans_freqs.columns = [c.replace("_cnt", "_freq") for c in trans_counts.columns]
        features = features.join(trans_counts).join(trans_freqs)

    features = features.reindex(all_ids).replace([np.inf, -np.inf], np.nan).fillna(0.0)
    features.index.name = GROUP_COL
    return features

features = build_aggregate_features(df, all_entity_ids, splits["train"])
y = pd.Series(labels_by_entity, name="target").reindex(features.index).astype(int)
print("Features:", features.shape)
features.head()


In [ ]:
# Cell 6 - Low-label CatBoost runs
iterations = 50 if SMOKE_RUN else 2000
od_wait = 20 if SMOKE_RUN else 100
run_rows = []
prediction_rows = []
feature_importance_rows = []

X_val = features.loc[splits["val"]]
y_val = y.loc[splits["val"]]
X_test = features.loc[splits["test"]]
y_test = y.loc[splits["test"]]

for fraction in LABEL_FRACTIONS:
    for seed in LABEL_SAMPLING_SEEDS:
        train_ids = sample_labeled_entities(splits["train"], labels_by_entity, fraction, seed)
        X_train = features.loc[train_ids]
        y_train = y.loc[train_ids]

        print(f"fraction={fraction:.2f} seed={seed} train_entities={len(train_ids)}")
        model = CatBoostClassifier(
            loss_function="Logloss",
            eval_metric="AUC",
            iterations=iterations,
            learning_rate=0.03,
            depth=6,
            l2_leaf_reg=5.0,
            random_seed=seed,
            auto_class_weights="Balanced",
            od_type="Iter",
            od_wait=od_wait,
            allow_writing_files=False,
            verbose=100,
        )
        model.fit(
            X_train,
            y_train,
            eval_set=(X_val, y_val),
            use_best_model=True,
        )

        feature_importance_rows.append(pd.DataFrame({
            "feature": X_train.columns,
            "importance": model.get_feature_importance(),
            "fraction": fraction,
            "seed": seed,
        }))

        test_proba = model.predict_proba(X_test)[:, 1]
        metrics = compute_binary_metrics(y_test.values, test_proba)
        row = {"baseline": "aggregates_catboost", "fraction": fraction, "seed": seed, **metrics}
        run_rows.append(row)
        print(row)

        run_pred = pd.DataFrame({
            GROUP_COL: X_test.index,
            "target": y_test.values,
            "proba": test_proba,
            "fraction": fraction,
            "seed": seed,
        })
        prediction_rows.append(run_pred)

result_rows = append_mean_std_rows(run_rows, LABEL_FRACTIONS)
results_df = save_results(result_rows, "results_aggregates_catboost")

predictions_df = pd.concat(prediction_rows, ignore_index=True)
pred_path = OUTPUT_DIR / "predictions_aggregates_catboost.csv"
predictions_df.to_csv(pred_path, index=False)
print(f"Saved: {pred_path}")

feature_importance_df = pd.concat(feature_importance_rows, ignore_index=True)
fi_path = OUTPUT_DIR / "feature_importance_aggregates_catboost.csv"
feature_importance_df.to_csv(fi_path, index=False)
print(f"Saved: {fi_path}")

try:
    display(results_df)
except NameError:
    print(results_df)
